# Olist – Sales Performance Dataset Extraction Pipeline

This notebook builds a consolidated `sales_performance.csv` file from raw Olist datasets.  
**Pipeline steps:** Load → Clean → Enrich (customers, products, sellers, geolocation) → Export.

---
## 1. Imports


In [1]:
import pandas as pd
import numpy as np

---
## 2. Load Raw Data

Load the six source tables required to build the performance dataset.

> **Note:** The `geolocation` table is **not loaded here**.  
> It contains multiple GPS coordinates per zip code prefix, which causes row explosion during a direct merge.  
> A deduplicated version is used in Section 7.


In [2]:
orders               = pd.read_csv("../Data/Raw/olist_orders_dataset.csv")
customers            = pd.read_csv("../Data/Raw/olist_customers_dataset.csv")
order_items          = pd.read_csv("../Data/Raw/olist_order_items_dataset.csv")
products             = pd.read_csv("../Data/Raw/olist_products_dataset.csv")
sellers              = pd.read_csv("../Data/Raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../Data/Raw/product_category_name_translation.csv")

print(f"orders       : {len(orders):,} rows")
print(f"customers    : {len(customers):,} rows")
print(f"order_items  : {len(order_items):,} rows")
print(f"products     : {len(products):,} rows")
print(f"sellers      : {len(sellers):,} rows")


orders       : 99,441 rows
customers    : 99,441 rows
order_items  : 112,650 rows
products     : 32,951 rows
sellers      : 3,095 rows


---
## 3. Merge: Orders × Order Items

Join orders with their line items via `order_id`.  
Orders without a matching product (`product_id` is null) are dropped — they are irrelevant to sales performance analysis.

In [3]:
orders_details = orders.merge(order_items, on="order_id", how="left")
orders_details = orders_details.dropna(subset=["product_id"])

orders_details = orders_details[[
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
]]

# Compute total revenue per line item
orders_details["revenue"] = orders_details["price"] + orders_details["freight_value"]

print(f"Rows after orders x order_items merge: {len(orders_details):,}")
orders_details.head()


Rows after orders x order_items merge: 112,650


,order_id,customer_id,order_status,order_purchase_timestamp,product_id,seller_id,price,freight_value,revenue
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62


---
## 4. Enrich with Customer Information

Add customer geographic data (city, state) via `customer_id`.  
`customer_unique_id` is kept instead of `customer_id` — it represents the actual buyer  
(one unique customer can place multiple orders under different `customer_id` values).


In [4]:
orders_customers = orders_details.merge(customers, on="customer_id", how="left")

orders_customers = orders_customers[[
    "order_id",
    "customer_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "revenue",
    "customer_city",
    "customer_state",
]]

print(f"Rows after adding customers: {len(orders_customers):,}")
orders_customers.head()


Rows after adding customers: 112,650


,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,product_id,seller_id,price,freight_value,revenue,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62,santo andre,SP


---
## 5. Enrich with Product Information

Join product data via `product_id` to get the product category.  
English category names are added from the translation table.  
Missing categories are filled with `"unknown"`.


In [5]:
order_products = orders_customers.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Add English category translation
order_products = order_products.merge(category_translation, on="product_category_name", how="left")

# Use English name; fall back to Portuguese name if no translation; then "unknown"
order_products["product_category_name"] = (
    order_products["product_category_name_english"]
    .fillna(order_products["product_category_name"])
    .fillna("unknown")
)
order_products.drop(columns=["product_category_name_english"], inplace=True)

print(f"Rows after adding products: {len(order_products):,}")
order_products["product_category_name"].value_counts().head(10)


Rows after adding products: 112,650


bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: product_category_name, dtype: int64

---
## 6. Enrich with Seller Information

Join seller data via `seller_id`.  
Seller location (city, state, zip code prefix) is already available directly in this table — no geolocation table needed at this stage.


In [6]:
orders_seller = order_products.merge(sellers, on="seller_id", how="left")

print(f"Rows after adding sellers: {len(orders_seller):,}")
orders_seller.head()


Rows after adding sellers: 112,650


,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,product_id,seller_id,price,freight_value,revenue,customer_city,customer_state,product_category_name,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71,sao paulo,SP,housewares,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46,barreiras,BA,perfumery,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12,vianopolis,GO,auto,14840,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20,sao goncalo do amarante,RN,pet_shop,31842,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62,santo andre,SP,stationery,8752,mogi das cruzes,SP


---
## 7. Enrich with GPS Coordinates (Geolocation)

### ⚠️Fix — Row Explosion (112K → 1M+ rows)

**Root cause:** The `geolocation` table contains **multiple GPS entries per zip code prefix**  
(typically ~9 coordinates per prefix). A direct merge without deduplication multiplied every row  
in `orders_seller` by the number of matching geolocation entries, inflating the dataset from  
~112K rows to ~1M rows and producing a CSV file exceeding 7 GB.

**Fix:** Deduplicate the `geolocation` table **before merging**, keeping only the first coordinate  
per `geolocation_zip_code_prefix`. This enforces a strict **1-to-1 join** and preserves the  
expected ~112K row count.


In [7]:
geolocation = pd.read_csv("../Data/Raw/olist_geolocation_dataset.csv")

print(f"Raw geolocation rows      : {len(geolocation):,}")
print(f"Unique zip code prefixes  : {geolocation['geolocation_zip_code_prefix'].nunique():,}")

# Deduplicate: keep only 1 GPS coordinate per zip code prefix
geo_unique = geolocation.drop_duplicates(subset=["geolocation_zip_code_prefix"], keep="first")
print(f"After deduplication       : {len(geo_unique):,} rows")

# Rename seller zip code to match the geolocation key before merging
orders_seller_geo = orders_seller.rename(
    columns={"seller_zip_code_prefix": "geolocation_zip_code_prefix"}
)

# Left join: preserves all orders even if the zip code is missing from geolocation
sales = orders_seller_geo.merge(
    geo_unique[[
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state"
    ]],
    on="geolocation_zip_code_prefix",
    how="left"
)

sales.info()


Raw geolocation rows      : 1,000,163
Unique zip code prefixes  : 19,015
After deduplication       : 19,015 rows
<class 'pandas.core.frame.DataFrame'>
Int64Index: 112650 entries, 0 to 112649
Data columns (total 18 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   order_id                     112650 non-null  object 
 1   customer_id                  112650 non-null  object 
 2   customer_unique_id           112650 non-null  object 
 3   order_status                 112650 non-null  object 
 4   order_purchase_timestamp     112650 non-null  object 
 5   product_id                   112650 non-null  object 
 6   seller_id                    112650 non-null  object 
 7   price                        112650 non-null  float64
 8   freight_value                112650 non-null  float64
 9   revenue                      112650 non-null  float64
 10  customer_city                112650 non-null  object 
 11  cust

In [8]:
sales.isnull().sum()

order_id                         0
customer_id                      0
customer_unique_id               0
order_status                     0
order_purchase_timestamp         0
product_id                       0
seller_id                        0
price                            0
freight_value                    0
revenue                          0
customer_city                    0
customer_state                   0
product_category_name            0
geolocation_zip_code_prefix      0
seller_city                      0
seller_state                     0
geolocation_city               253
geolocation_state              253
dtype: int64

Delete the last null values from geolocations that represent ~0.2% from the data ()

In [9]:
sales = sales.dropna(subset=['geolocation_city'])

---
## 8. Data Quality Check & Final Cleanup

Verify null values and remove any duplicated columns before export.

> **Fix applied:** The original notebook called `sales.reset_index(inplace=True)`,  
> which added an unnecessary `index` column to the CSV.  
> This is replaced by `index=False` directly in `to_csv()`.


In [10]:
# Remove any duplicated columns that may have been introduced during merges
sales = sales.loc[:, ~sales.columns.duplicated()]

# Fill remaining null values in product category
sales["product_category_name"] = sales["product_category_name"].fillna("unknown")

print("Null values per column:")
print(sales.isnull().sum())
print(f"\nFinal dataset shape: {sales.shape}")


Null values per column:
order_id                       0
customer_id                    0
customer_unique_id             0
order_status                   0
order_purchase_timestamp       0
product_id                     0
seller_id                      0
price                          0
freight_value                  0
revenue                        0
customer_city                  0
customer_state                 0
product_category_name          0
geolocation_zip_code_prefix    0
seller_city                    0
seller_state                   0
geolocation_city               0
geolocation_state              0
dtype: int64

Final dataset shape: (112397, 18)


---
## 9. Add Country Column for Geolocation Mapping

Tableau Public cannot automatically distinguish Brazilian state abbreviations (SP, RJ, MG...)
from US state abbreviations. Adding an explicit `country` column with the value `"Brazil"`
forces Tableau to correctly map all geographic data to Brazil.

This column must be assigned the **Geographic Role → Country/Region** in Tableau
before building the map.

In [11]:
# Add explicit country column to allow correct geolocation in Tableau Public
sales["country"] = "Brazil"
sales.head(5)

,order_id,customer_id,customer_unique_id,order_status,order_purchase_timestamp,product_id,seller_id,price,freight_value,revenue,customer_city,customer_state,product_category_name,geolocation_zip_code_prefix,seller_city,seller_state,geolocation_city,geolocation_state,country
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,38.71,sao paulo,SP,housewares,9350,maua,SP,maua,SP,Brazil
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,141.46,barreiras,BA,perfumery,31570,belo horizonte,SP,belo horizonte,MG,Brazil
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,179.12,vianopolis,GO,auto,14840,guariba,SP,guariba,SP,Brazil
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,72.20,sao goncalo do amarante,RN,pet_shop,31842,belo horizonte,MG,belo horizonte,MG,Brazil
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,28.62,santo andre,SP,stationery,8752,mogi das cruzes,SP,mogi das cruzes,SP,Brazil


---
## 10. Export to CSV

Export the final consolidated dataset.  




In [12]:
output_path = "../Data/Processed/sales_performance.csv"
sales.to_csv(output_path, index=False)
